# MatMul Bench — Web UI thực nghiệm nhân ma trận trên Google Colab Pro

Notebook **tất cả trong một** (đã nhúng sẵn `index.html`, `server.py`, `colab_tunnel.py`): chỉ cần upload file `.ipynb` này lên Google Colab Pro, chọn **Runtime → Change runtime type → GPU (T4 / L4 / A100)** rồi chạy lần lượt các cell bên dưới.

## Bước 1: Kiểm tra phần cứng GPU/CPU, clone repo & biên dịch CUDA

In [ ]:
import os, subprocess

# 1. Clone hoặc cập nhật repo
if not os.path.exists('/content/Matrix_Multiplication'):
    !git clone https://github.com/dthai2424/Matrix_Multiplication.git /content/Matrix_Multiplication
else:
    !git -C /content/Matrix_Multiplication pull

%cd /content/Matrix_Multiplication

# 2. Kiểm tra GPU & tự động nhận Compute Capability (T4=sm_75, A100=sm_80, L4=sm_89)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!echo "CPU vCores: $(nproc)"

cap = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True
).strip().splitlines()[0].replace('.', '')
cuda_arch = f'sm_{cap}'
print(f'>>> Biên dịch với CUDA_ENABLED=1 CUDA_ARCH={cuda_arch}')
!make clean && make CUDA_ENABLED=1 CUDA_ARCH={cuda_arch}

## Bước 2: Giải nén bộ Web UI (`index.html`, `server.py`, `colab_tunnel.py`) & cài FastAPI

In [ ]:
import base64, gzip
from pathlib import Path

web_dir = Path('/content/Matrix_Multiplication/web_ui')
web_dir.mkdir(parents=True, exist_ok=True)

FILES_B64 = {
    "server.py": "H4sIAEk2CGsC/9U8y24jSXJ3fkW62vAWe6gSKWkabcIcWC1xprWjlyX2eBeUXFusSpI1qtfUQxKt1Xlh+OKbD754sPDNBhawAQPbxzb2P/pPHJGPqqwXRfX0HFaDkcjMiMjIyMh4ZWa/+IvtLIm3Z26wTYNbEq3SZRjsdjRN63xtJen++RGZWfYNDRwyD2NyYqWxe09OMi91I8+1rdQNA/KGBvbSt+Ib8vdhfDPDb0T/JgwXHiUHoWfNyBdkkgUB9dxg0TU6F1mQEGPbt1I/88yZxO6RJI2p5SfEc28pfHHCLN2GPzSOyV+Rt6tZ7DrE8qKl9ZKkWQDEyK1rkcvLca9jAYcJjW9pQmKaAH8JDGrYyS3ZJsa9l+CfZep7xAnvAi+0nMRgs+zM49AnpjnP0iympklcPwrjlFhBEKZsekmnI9qAmvz4fRIG8nOYyE+RZ6UgJl9+/yGjGZVf4vxTks2iOLRpkiOmS5i4AxPKG1w/B88y1+FsRla69NyZ5PEcvvKOdBWhNET7frDqkUPXTnvk2E3g91mEE7E8Mds5rKwVuRJcLHSPvJ1Mzsf3NmXQJVDDdx3Ho3dWTA07jBOJenB2cXmSd5VRYBkikB7Ngb92PXohGnGwk+Pi2yVbeZiDbBITXjlWkLq2JPHGSuhJ6FCvB9So53Q6b/Yvx+bh0QUZMXHosJIwjGl2cfzQu6V614iAtyDtXIzPz1TQMDFA5904DIwFTXXtZH9y8u7YlGBaj2iG1lUIddw5Aa0gek5pm2hVLda6Br0HqSd6l6BS6pJDwcYTOMMOgR+F1Qo6zPjodP/i1+bp/skYumu0gBLVCHAK0wssn5IRAAWpRqiX0IahO0enh+NfmbgcymjIpRs49J5tGtgmVhRBt1AVPXVTj440MAdgCvj+J9AOIrulcQLaM9IGRt/oa11ENCzHMQsN0tkUy6rTY22W54V3Zhi7CzdIRlPtpXatdvgUrJPT0LGEzQPjyo5up9OxPStJCFiaCwq7MEn1XHWEiBP3H2kyZBtk6gbpNc4OVUp36NwC+zGafjnY6ZFBf2evR3b6e697ZK//16+uewTU1PRosEiXowF8s+7zbztdRprv5vXEAROoA2mgO9iIagxWc0iAWo3Ybo8sKGLBmuz0ObSdOZaZwlZoRhm84mCLKDPDwFsNySwMvRrY1xYojRieJjQ1l8wIm/Z8sR5DLkDuGn4ZzrjcARDMrRu4qWnqCfXmPfJ9ODNdZ4jmv0ds3xGCg68gF/sOvuN+FevG1g7QDI4FDPAP5U6gAj3wu9J8x5rvKs30FnYWCFdjBtv4O/w9lUZzipZ0yngDw3p9fa0BCQVQ75aJCdM+zK3utLD3xnkY0QDV4TQMaIU3K7Cp51GH7TOQYrl7DiJLlm29YD5S0wb1VoYVilcfSSwiVygm6soUAWt6XcYx7SxG+yM0QBmmjstG7HR+efbmckiKblUXEOzhkYGYx2cH38LX3Acax6F9o6MOvSBbn+8HqL21YgetDUQT6IxArcBYgd98S70I7MdnHg81HRyLCcGVFa9MdN7gE7a+YtrMlRljKlh3x3WslMJGLbkWxdD3VMNcakdaqNZAudSj7BbwBfkYFT8jf2IK0U9QgHWUxmaOYHVwfg5NqQ1KIQQL+3oe8kmW1YIPh73DSg8qQs6LFibakMy1BxlJGckqSamvdx9J0RZTj4I1h0atV6DaUQa4OZDYcWEM7ICUtXfBTQChHzk4f6dieeECwlgP9g4YOMAHrwmE4GsWpBxzoEBHy1VSAgc7rYPh1etoO7Ag2/BbwY4t31zMAAs8o9K8YIxr3y4//A/oYrT88GNKlu7H978LyDfn78jBu8N9leNbJOMzMursQz/KUmraVoTESoJBVyAZVnGEYuK2W0FXTVdzZVHnQKPQdNxYY+Y6V1cB8thhf16AVQmye3J2STAE6aHQic/jNilCwjjqkYv9k47QUhaVads0tbfDZEusci02Yh42XpUVGDcSZBdsD7WRwXmaKb1PdTBEIVqakZal863XELZAigFh7UhzFwGwBcAJpDcpUqztFcEs9hkQ6cZpcufigOcX48mEb46R1oAjN8AUlRzVnlPAcXRtBDwMutPBNZCM3Qhkz//+QvtFt5HSDGZzk/dQFrWDvMc4j/LYEfjhTkXEuDm2QWGRn80EjHKTgXMN/zmSLVEFRUatAMKQWCTUiu2lHmv/wBtRda6Sl0P4Xze+6P4lEMMxeggLgfrk6PjodFymBzMUJBvWjMkerQQKX4AZizjMIn0g5V1ljylplT2gwbVXcnflPIM9htnGXsXAIKfgxnWBlXPb/Wkr71P/J618gf8TVh7MWFWwJ9SfQNbt5UK9Sr64ebOpYIFgm1iF5UVxggQDR+dCheZCpODfdIz2yUsW9HchQH+umF+Q0++ODo/2md3G0gSkl45rbSW+26nJNsxQrEpsaC+pfWNCMxhyvTTCtDYtraCsmPq8d2sLAtR4tQWuZcQsMKxYGK+MFKXbU31FIzI6UCsd2cltLwh5cgUfMojZkwrCdfkrL9aMlEkdjr87fXd83Kup1GgSZ7TS7PoUpj/aK5rr+xIz2yytWH83TlCY00iCM4cQoTcA4JIxn/avpdHtad3rqhpB0qUzcl3y1YjstinUQpgRBgokW8Ckr5b7eO6FVsrpg63vdtuMlLJAxSg7fBShinmhpmCxUEQRuiG1cqBmeYvQvKErHbWCuXAWrMFfkRXDaKyPxUywwSG9prEQP4hHSzMrIGmmoWgTBjQD74qyVZoS+gP/VvAmOGJdOTE7JDMrKNNy7TZUEIrJAXICsxXOeS0Chyj4h7jfESNieYZFRxor67QT8XZMb7egcUMhQAsjhW+eFLXxIXpz/IUXziyvDZqFawIkR0kgwqbOWhQB0mkRGVeDCDIeaooKqQkbXIf/WbgnkmzUhqaUcChZQUlJnAb3IUYW6WNMwW85T2aZJbuI0VQxAubLuhajOyk7mK3EXUBrQO9Q+Uagq8RKyLya16DtwpQ/uTVw+AvWoM+7tejRde7Bw4R3KGIaZD6NIQ/SOQGsTEOgNxo0hHU1fyl/AvPW8kr7HqjzSuO3rr2cLLPQNk81lizkPeJ7v9sc9YnKUjPVy/A4C4OFyX//UKE84ai8ddBCn5kHXjUkuoKaWekkhJ3alWahKVrKmTQTaqPVKrM3WYbuN64VmIIHSIKa8ReAGCV1Am9dmp1aMLVvvj4+O798gkoSUepkUQMfQGICor/kAEIgbWRuaezOXVZ20eaW6+WmQl1N6h+6wVJI53z/8nJ8WDGdDQkE14+vSJ/ZIS40+NasTMpmMqwI9oSjt8Lhz8PaXubmwSINmdY/DRoAJGN3A1ihooAhPm2AAw4J4KsuKlfG7gYkEBho5DibcArhBrKJkt8AnKsl5snswwYYQgUxVeafNsCR+gZI8uN6rMfW3rrmidDhO8vLaEMgK3/sMEghf6eddZFv1dCzb0JFS86mKBnr8P/zfY3AedLX0ABsEn2er1HLBvlAn6lOwOjKLL/BXIr5MTDBRqme8KKpiFBampxSSv2mstpjzcml4Q1OVa08/FZrrm9oIxZtAEazktz0yC1WbcObcg2j2dwBh9MbKQZk7rZNJho74GVjs3nVlZiv81oz2G7+RJkQSXPrjd976NPat5mo0BU4iw1wCjuIrrrAle0oq3X4+QFOjULRg0dHa4lwWQ6FCyxI8PYeeM/Xa/FdH6JQPJdooFH09Zj7bKFTt0/dJ9KYRssiVp0dClzQBb2nCYEc1V5i/T6J7W20/DFsHB8rsREaz1mYUHlrgGfVnYux+fbXb8zLyf7FhBcfMNUCUeqxdghxAXFCsoQwgyDdJSSkkA/iZ4uklkuudFaUYNXMqy/IgKX5fD17ZIIHCrxo0dW6xUjj80F5JD4r7Q2Ef0DiSj8dCawhIwyB1/TKMa5hHB7n4LCiJbnqkqvfsmGfANM6JRZ2qpNlo+8Azj5qQh8HZzrRVwiXZ7HbPotdoPMdxJwkn8iJCTzyefXgy4H8gtynvE/y2oOGA7UBYY6kbuXNyoQOz07H1fkcLMGUshlcvVRlc6XXSanrc7B/8HZ8WKV2akXPItbg7vjtFRMtrf59OBuWTr96zAIX6XfbWRrfE1gmUxVXFsyQRp6YK4YShqse15UOWaQZNMsmSimF9TprTZGE3KlDWp47MzFcZOfm5c6IxnOTm188Iaz0piY7QIGkt7GbIS/akRfrkLlqN/eBQYv5PR/on143YZrolluwFQPZ0I2XWIAte4n97Ni2AHisGryGdevUFQDsSbMCsDSiae3dhAUayN5aJTGyCM/+yh617kmVRV6rNtU15/6jVXdqSlCG320nv2ggv9dCftFC/ssqfLE63U9dp52faZ2mUp+v8+TWbzgZeCa3uz8bt8oOu26M2hp0jFXpN9IwX6z/WsMkIO0a5G67nmysU03q/WUjbD2ckvCvPrP+oY/8edWPG8X1KtisDrkQ6sg7FWTlporUnAaaygWcwMKKjwNudD1gs48sXZUR0kaSNfHysOEpR4wx66rueks+pSa8NQ7mCQta9jb145V2V/f4lNDZTGqKyFpLBw7i5g9GQ3dhfENjpFKPf1jQUyhaKSXHg6PyuRi7M1W2GMim7VcqI/adM8I7CazzzunWjqbwbEmle3Q+fur46nJyePZustHp1Syb453C0aDcLCsGMY08C5Olho2NHIsRYer4Ke+ykoTizWJoM/gU1I1aKmTE1p0pixkKfGNVQsIaMU/Dtav4Kqgc0SJX/GacgSeSD1q6ilCz8M4Mpr/s0GfIKD6WMUGDTB5M4HTWhMU8EK7VABT8Rqv0JIfixAWYBCKYABcUgdWSrhl3lpvq5bXIb9OJ1TC4emNLY+6KJw/QNmwlsjXoPFOqc206vrg4u7gmD0D5URMSnruQJnir8kjK1UBUy7wPD1FERS5fB/XoR2wUvOub3weWAHh1XtEHLI6JFLy6pFjYUyiJVmbkDOjTum1TfyoGkIJxYO2bzqlzAQNQSeC9pphVXK4UsPn3BlghMoBUBNgAV8wfQQsBtYOiSRUMKC0N8HiHHSGb61gars2QbLB+DTe4SoTuveRJQgjzJCF2S7yNkr3Eqia/Sf4kJS7O9Rq1hsbj02FURQ3RtPwcl03x7cwFGDv6ue+V/i1eqWcluG0NL8XwBxMmu3A9Up9VdJkjZq9izMyV9WlRdy6u/TdV1i0Xa2fqexAdApY0S9gGG+3193p4XmO53kh5J8AIz1no3lXDApUpXRm4vdLexRVRJmpF7ra8ZKrxeUFTfu9U8r68w+iv8U5qR4QIAFC/6CjeVigsK3dSfW5iNHyVpN6rzPkB93Kntqt2oX4G0qbUCoXm7S+3/SdtebnVP2mb59v7k7a23NLP2c6P+X3SqhYIfhUlEC1SBz5pjQub33BDYq28P9fC51ONwiSfaxao84RQOqY/DJVXLWLKfKJNs1Z3PO/YcLdX4mJ16/fLyyvMwFw7WP7pDxZJP/ynT9Llxz/+x0ry9cD/Phrk7Yffr4gNnT+uyG9864ay283m+HT/zfH4cDT4DeQCf/rDx/f/ZhulQFlOA6Zv5NXI/CR+8KpHdneeZb76hfkq6AFj7/+JBMDefwVk8Iosw49//F8bSBuaYMH2KF5gwKdDmKOEcUod/SFg4Tdet2IM8m7g9tUe+ZsRtMOvwavd13sihuNEiqsckkzKj+gkGQkAhAZIImV0dl4/lhZV5QiZyJvkI6RPE8qhFSxJ8uFHe0lOye2Hfyfi+gi54bfUQVK/j4j38f3vCtmwfSeUTNlsYtlq74dKsSVaPP4yp3GPlN4yCPCWpwyN93FynCyA+Pqmcuq47nJn+V4df1ZUXMfEdJPPWLUDW6eqm+hpxvehi7KO9Xt+MfGenTcXK1dCTjdHFmtcQo8VdMRisocoswS0hUq/xc8Qa9D5hiihhCpbzbaQA1yrq56/LytW23dkVQH4gP4t7JdKlD/qwkevBv7aA9O9pPfT4UDcfwQQfKiolBN0jjXif9gTshFWB1hFACsBnfxeWf7gqGAIm6Yc9Zq/Jusor/jwPRLXfT21YrCvo6Kw0SPQkoxYItuFvWNRPwxYZaDLT/L1UhT0oPFReAbAOMWbnr7FKp0aEUsNnHcb/B5/G739wFEfFb/Ae3TlFZ3YEm0z5hLEDkafI+ZWBXsh6y5n3M8KCMVLFtUTYMwt1xj5ZvG3uaABVqTwjY5y/2+JtgCluMHGxuNovmgypMcZyYvEg9rtMLHV+fO9sR+lDTRfkG8pjSz2CB3XBx/tRjEnTw68MHPmHj4i40/ayaDfByPtAM9i2BrBFb6QJLDGN5LuVXAVaE23LtR0frjBw4/190LmXD611WynxVmds7IJxJ/4zN1wMj9K2ME/XsJM8Jm8ldiuK956PvLJqJpee81dhBO1hS+sik8dFwwPZP0jDZOCbQa7xdVbsT7ytW8lOj7AwufWAQgiDjFY1YJwixdDK+Hor7b2bUj+t95k8zmNgUsOrNWiz25DSMYLB03bkPf8WWzDvISqPDnNK0dCC2U9Uh4XyIpoFHqe3q2zVtueKlJKY1AHPFx8xpOKkulUCzjIaoOFlP+yw/bD3E/VlZEdOrSrC7NBpuDDEHjDpPRIUeRhbV6QcO2tZgciAWtJvQALhpL/pMb2beAYfrKFMvLKOYbIxFpyMDk4T8jUR4A8B6uHVhIj8iw3KJIRoQkgMRlmC1F8ojL+3798fP/PEFM6EPuDPIsw8l9BcyCYlGqJURp/M1iYA3yuxQefAj/iGh9zxnnIKNHU0JiD/MTqhkxr7A//zQYhD3KoSjqTr4eSwpTcv/rPX+ioaeywgjMJ0YMkO2qSwKj4iEbJxSf0CGOa7J93ME0fFs80NVHhEf9uyK0LSW3ALTNr4dfGq//uxfnZxQTV4HW/39fEAZzAxchRB9H3IBdK0pHWN9h/AI30Rvir2/l/HDZcFERGAAA=",
    "index.html": "...",
    "colab_tunnel.py": "H4sIAEk2CGsC/7VWzW4bNxC+71NMt5dVYlGy3fxUgAo4sZMYMWLBlhGgtrFY7XItIlxyQ3IdK0GAAnmB5tACPRRNG/TSokgPOVlHBX0PvUmHovZHcYKeSh3EJYfz930z5JdfdAqtOiMmOlScQz4xYyk2Pd/3vX1B27HMskgkwKNCxGOqIJUK7kt5xinclTwawUDJnrdO4NBEymi4F2mzNdiFQFN1ThXJJy2QAnKpDNzudrvE2yCwLZ8JLqNEQ8xlkaQ8UjQBlkLGtGbiDKxJmVOhIYJUUQp5MeIshgfD4eAQTCEE5cTbJGidCbRqxpXI2Jhc9zqda8SoSa2eYCRwdLAHkQYt0SP8ZwaYBkWjZEIWEXssWzgqdTlTtJzpcWEYr76KUa5kTHUlqSfV1LCsOlYoztmIKPq0oNp4qUI38siMcRGWIgP89Lw7W4c74fbuAfQXC0EYpozTMGzhWS35OQ1aJMdAhPEG+wdDFMPIA6kJwsaUFOSMmsC3W/4a+DbVfqvl3d3bP9q+t7d1sLMdDraGD0rt/gJ1LuOIL7Bv4OCvnrI560PgAQ6/TO4ZM+NiZHPaONlU0lGU00hT3eGRwcA7yRLyplCbM1FctKMsufmV77U8z0toCoh6oWjYkAta0P4GtFG9hRepLJAe/SUi5NmYxRjQagRWDvm0EHWn7FDUFEq41cUichbThrqu5AkPY26j2EIcfLy7BGLNijwO9x+2gHJNoYKwAyvulN4IaZYWCb1g2uigVfuWWyYH/vF6Z/MUPnwfYRWY+eVb1iwRQsgyNjtWqUXwE+NTjCJVPgJwbWm2Prt0Ix5nMgm68taNG25vmSHMdVAecahkERMOh0dSUOd2nIbIHczepyDzvDqq1D/esGE9HM+nvzD48Ho+/QnjK3uFaxVg1OwPAfF8+gPuvbBMflkH7GRCW3QW+qr8yMD2iaAK7BjrELNL48JEI06xFNqZLYjinMVSCTt1qnpRntuvdnsstbGz9Y1bpIu/dbdsaxNnNhXWl9bpWmUkfpb07XoJeKve0iaRhek3Hdwd7KzsU6Wa+4fD7f2jYS1h6IXpD1VB3ZIL37YUojmlebBBuhW/G1khueQcEcKOZnlWo2QHumSz1pB2fhLb/Oyh9BObjtS+f4Wie/Ppj7unFXxPxrP3iNiTVXA/vP7n7/n0bdw7sUlHdTX5HETMBOsrLPGPN2vuZ6gM7laEguGi50MQo5Wf4Xz2hsEZm/02Wdh7BWI8v/xLADaUJ+6SaNXUQZr+N20clxF4d7k4CgjZjgojizzBJuaWsMpwki46ITbCijM9R1j/9H+kgh2jItXsOe2vl+zwlq0gxCT1wV10Od4cgfKDsl0fR+3nW+1vu+2v26fXT1YvxhN7oLVMlbtDQ1SHuiyFHPnUpKZSpJEppkxqSZUG6ypJ+1RAQCheVB+J19rsyNCWC4BoGils5vZUa0XGvg0WrwJrpfZyVdGVCDJyphA9y7Mrco5ySE64Dn7fh2tw8/bnxFIf4PHOHTjaRXbOXsHh/PL9Izicfffofg/gRW3zpf9ZSwD32Xz6DmLKOYjZG8tcS3Qk9OWvE8yxxA9dzKevDfDZZQwJzSSBO/PL3zN8XMl8yXS8FN4Z8nk7y1BsVIvKS3mhxwsSuSP0Iqa5gYd0MpKRSnaFQRIWublyE50IV4nJfPqndc01aay88v1VFVjKRMR5gyMl2qg6wz2Db5ea9Y0+09zHx1cKYSiiDB890O+DH4b20glD3yl2N5D3LyZHYZCxCgAA",
}

for fname, b64_data in FILES_B64.items():
    content = gzip.decompress(base64.b64decode(b64_data))
    (web_dir / fname).write_bytes(content)
    print(f'[OK] Đã tạo {web_dir / fname} ({len(content):,} bytes)')

!pip install -q fastapi uvicorn
print('>>> Đã cài xong FastAPI & Uvicorn!')

## Bước 3: Khởi động Web UI & mở link HTTPS công khai (Cloudflare Tunnel)

Chờ vài giây, cell sẽ in ra đường link `https://xxxx.trycloudflare.com`. Mở link đó trên máy tính hoặc điện thoại để chạy thực nghiệm trực tiếp trên GPU Colab Pro (giữ cell này chạy trong suốt lúc demo).

In [ ]:
%cd /content/Matrix_Multiplication
!MATMUL_REPO_DIR=/content/Matrix_Multiplication python web_ui/colab_tunnel.py

## Bước 4 (Tùy chọn): Tải trực tiếp file kết quả về máy sau khi dừng tunnel

In [ ]:
from google.colab import files
import os

for fname in ['benchmark_results.xls', 'benchmark_results.csv', 'benchmark_charts.html', 'hybrid_alpha.cfg']:
    path = os.path.join('/content/Matrix_Multiplication', fname)
    if os.path.exists(path):
        print(f'Đang tải {fname}...')
        files.download(path)
    else:
        print(f'Chưa có {fname} (hãy chạy benchmark trên Web UI trước).')